# EOD_GAP 종가매수 진단 — 2026-08-27

**결론:** 1차 수정 네 가지는 코드에 들어갔지만 완성되지 않았습니다. A/B 후보가 후단 게이트에서 모두 탈락할 때 일반 후보로 넘어가지 않고, A·B 중복 후보가 남아 있으며, 일반 폴백의 거래대금배수 3 조건과 상한가 레거시 설정도 남아 있습니다. 라이브는 `NO`입니다.

범위: 2026-08-14~2026-08-27, Asia/Seoul. 이 노트북은 손익을 계산하지 않으며 매수 퍼널의 차단 단계만 집계합니다.

## Context & Methods

생산 실행 로그, 체결 CSV, 체결확인 로그, 현재 생산 코드와 Windows 예약작업 상태를 대조했습니다. 일자별 마지막 차단 원인을 상호 배타적인 4단계로 분류했습니다. 체결 부재는 브로커 체결이 아니므로 `[UNVERIFIED]`로만 표현합니다.

In [1]:
diagnosis = [
    ('2026-08-14', '후보/선정 조건', 'A후보 선택 후 최종 점수 게이트 차단'),
    ('2026-08-17', '후보/선정 조건', 'A/B 후보 없음'),
    ('2026-08-18', '주문 안전조건/브로커 거절', '주문 status=ERROR; 세부 원인 미보존'),
    ('2026-08-19', '주문 안전조건/브로커 거절', '저가 후보가 전역 가격 하한에 반복 차단'),
    ('2026-08-20', '데이터 준비', '실시간 보드 오류 및 당일 캐시 없음'),
    ('2026-08-21', '데이터 준비', 'EOD 데이터 신선도 차단'),
    ('2026-08-24', '주문 안전조건/브로커 거절', '시총 하한 불일치로 게이트웨이 거절'),
    ('2026-08-25', '주문 안전조건/브로커 거절', '전역 가격 하한 거절'),
    ('2026-08-26', '주문 안전조건/브로커 거절', '전역 가격 하한 거절'),
    ('2026-08-27', '접수 후 미체결(잠김)', '잠긴 상한가 주문 접수 후 미체결; 대기 게이트가 대체 차단'),
]
from collections import Counter
counts = Counter(stage for _, stage, _ in diagnosis)
assert len(diagnosis) == 10 and sum(counts.values()) == 10
print('분석일수:', len(diagnosis))
print('분류합계:', sum(counts.values()))
for stage in ['주문 안전조건/브로커 거절', '후보/선정 조건', '데이터 준비', '접수 후 미체결(잠김)']:
    n = counts[stage]
    print(f'{stage}: {n}일 ({n/len(diagnosis):.0%})')

분석일수: 10
분류합계: 10
주문 안전조건/브로커 거절: 5일 (50%)
후보/선정 조건: 2일 (20%)
데이터 준비: 2일 (20%)
접수 후 미체결(잠김): 1일 (10%)


## Results

1. **적용됨:** 상한가 우선 OFF, A/B 공통점수 중복차단 제거, 가격·시총 사전검사, A/B 주문가능 후보가 없을 때 일반 폴백, 당일 캐시 10분, 1주·1포지션.
2. **빠짐:** A/B 후보가 정배열·스마트머니 등 후단 게이트에서 모두 실패하면 일반 후보를 이어서 보지 않습니다.
3. **빠짐:** 같은 종목이 전략 A와 B에 동시에 걸리면 후보가 중복되어 실패 시 같은 종목을 재시도할 수 있습니다.
4. **남은 레거시:** `LIMITUP_ADD=YES`, 만료된 `LOCKED_D2_PRIORITY=YES`가 상한가 매수를 끈 뒤에도 남아 있습니다.
5. **남은 병목:** 일반 폴백은 `MIN_SCORE>=70`에 더해 `VR_MIN>=3`, `DR_CEIL<15%`, 정배열 strict를 모두 요구합니다. 최근 실제 저장 데이터 재구성에서는 거래대금배수 3이 일반 폴백을 막는 핵심 조건이었습니다. `[HYPOTHETICAL]`
6. **정상:** 체결확인과 익일 매도 복구 경로의 집중 테스트는 통과했고 현재 EOD_GAP OPEN/PENDING 상태는 없습니다.

In [2]:
gate_review = {
    'applied': ['locked_first_off', 'setup_score_bypass', 'orderability_alignment', 'empty_setup_fallback', 'cache_600s', 'qty1_maxpos1'],
    'missing': ['post_gate_general_fallback', 'strategy_ab_dedup'],
    'legacy': ['limitup_add_on', 'expired_locked_d2_on'],
    'blocker': ['general_vr_min_3'],
    'live': False,
}
assert len(gate_review['applied']) == 6
assert len(gate_review['missing']) == 2
print(f"applied={len(gate_review['applied'])} missing={len(gate_review['missing'])} legacy={len(gate_review['legacy'])} blocker={len(gate_review['blocker'])} live={gate_review['live']}")

applied=6 missing=2 legacy=2 blocker=1 live=False


## Caveats, Assumptions & Sources

- 2026-08-18의 `status=ERROR` 세부 응답이 로그에 완전 보존되지 않아 하위 원인은 미확정입니다.
- `DATA/eod_gap_positions.json`의 GHOST/CLOSED 상태는 일부 체결 CSV와 충돌하여 체결 원장으로 사용하지 않았습니다.
- 주요 소스: `data/LOG/eod_gap_live.log`, `LOG/fills_202608*.csv`, `data/LOG/sched_EOD_GAP_FILLCHECK.log`, `RUN/eod_gap_live_executor_v1.py`, `RUN/hidden/SAFEPLUS_EOD_GAP_PICK.cmd`.
- 이 결과는 진단 집계이며 `[PROD_REPLAY]`가 아닙니다. 라이브 조건 변경 전 현재 생산경로의 보존입력 재생이 필요합니다.